In [26]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [27]:
#Imports:
from ncps.wirings import AutoNCP
from ncps.torch import CfC
import pytorch_lightning as pl
from pytorch_lightning.loggers import CSVLogger
from numpy import genfromtxt
import numpy as np
import torch
import torch.utils.data as data
import matplotlib as plt
import torch.nn as nn
import os
import time
import json
import csv
import sys


import time
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parents[0]))

from utils.functions import create_simple_dataset
from utils.models import lstmOBU, OutLogger

torch.set_float32_matmul_precision("high")

pl.seed_everything(1000)

Seed set to 1000


1000

In [ ]:
# Step 1: Define variables
data_file = "../data/RandomPos_0709.csv"

In [ ]:
# Hyperparameters
batch_size = 64
lr = 0.001
epochs = 1

# OBU

In [ ]:
# Step 2: Create Dataset
# Generate Time sequences that are 10 timepoints (Messages) with 7 features per message.
# Organized by car.

raw_data_set = genfromtxt(data_file, delimiter=',')

# Ceate dataloader and fill with (BSM, attk#). Expanding to add 0th dimension for batches.
# Batch size should be 64 for the low density simulations and 128 for high density simulations.
# No shuffle to keep batches on same vehicle.
# Num_workers is set to = num CPU cores
raw_data_set[0:-1,:] = raw_data_set[1:,:] # Get rid of the first null value of the dataset

# count subsets per vehicle
unq, counts = np.unique(raw_data_set[:, 2], return_counts = True)
sender = 0
last_sender_count = 0
new_data = []

# Organize dataset into sets of 10 messages by sender
while sender < counts.shape[0]:
    # Loop through sender
    index = 0
    while index < counts[sender] - 10:
        # Loop through messages from sender
        new_data.append(raw_data_set[last_sender_count+index:last_sender_count +index+10])
        index += 5
    sender += 1
    last_sender_count += counts[sender-1]
data_set = torch.tensor(new_data)
leng = data_set.shape[0]
train_perc = 80

# Create new arrays per vehicle for federated learning
splits = np.split(data_set, np.cumsum(counts)[:-1])

# Create seperate datasets for testing and training, using Train Percentage as metric for split
x_train = torch.Tensor(data_set[:int(leng*(train_perc/100)),:,3:10]).float() # 1
y_train = torch.Tensor(np.int_(data_set[:int(leng*(train_perc/100)),:,11])).long()
x_test = torch.Tensor(data_set[int(leng*(train_perc/100)):,:,3:10]).float() # 1
y_test = torch.Tensor(np.int_(data_set[int(leng*(train_perc/100)):,:,11])).long()

x_sm_set = []
y_sm_set = []

# Create dataset of 1/100th of the entries for quicker testing during development
for index in range(0,int(leng * (train_perc/100))):
    if not (int(index/10) % 10):
        x_sm_set.append(data_set[index,:,3:10]) # 1
        y_sm_set.append((data_set[index,:,11]))

x_sm_train = torch.Tensor(np.array(x_sm_set)).float()
y_sm_train = torch.Tensor(np.array(y_sm_set)).long()

# Create Dataloaders for all the datasets
train_loader = data.DataLoader(data.TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=False, num_workers=10, persistent_workers = True, drop_last= True)
test_loader = data.DataLoader(data.TensorDataset(x_test, y_test), batch_size=batch_size, shuffle=False, num_workers=10, persistent_workers = True, drop_last= True)
train_sm_loader = data.DataLoader(data.TensorDataset(x_sm_train, y_sm_train), batch_size=batch_size, shuffle = False, num_workers=10, persistent_workers = True, drop_last= True)

data_set.shape

torch.Size([623867, 10, 12])

In [41]:
# Check shape
x_train.shape

torch.Size([499093, 10, 7])

In [43]:
# Check: should be 960.0
tom = []
for line in data_set:  
    tom.append(line.nbytes)
if (np.average(tom) == 960.0):
    print("Check passed")

Check passed


In [ ]:
epochs = 1
lr = 0.001
test_obu = lstmOBU(
    inputSize = 7, # 9  # Number of features per BSM
    units = 20, # Number of hidden cells
    motors = 8, # Number of motor neurons
    outputs = 20, # Number of possible labels
    lr = lr, # 0.001
    gpu = False
)
path = f"saved_models/Normal/cnn-{epochs}-{lr}/"

log = OutLogger(path)

GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
/home/sliu/Documents/GitHub/reu26/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [ ]:
print("=== Test results before training: ===")
test_obu.test(x_test, y_test)

Test results before training:
torch.Size([124774, 10, 20])
Model could not complete tests.


'Model could not complete tests, found 0 of misbehaviour.'

In [50]:
test_obu.train(epochs=epochs, 
               dataLoader=train_sm_loader, 
               log=log)

/home/sliu/Documents/GitHub/reu26/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.

  | Name     | Type             | Params | Mode  | FLOPs
--------------------------------------------------------------
0 | model    | CNNLSTM          | 1.9 M  | train | 0    
1 | lossFunc | CrossEntropyLoss | 0      | train | 0    
--------------------------------------------------------------
1.9 M     Trainable params
0         Non-trainable params
1.9 M     Total params
7.425     Total estimated model params size (MB)
5         Modules in train mode
0         Modules in eval mode
0         Total Flops
/home/sliu/Documents/GitHub/reu26/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 0: 100%|██████████| 779/779 [04:30<00:00,  2.88it/s, v_num=0, trainLoss=0.000698]

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 779/779 [04:30<00:00,  2.88it/s, v_num=0, trainLoss=0.000698]


TypeError: OutLogger.updateLogs() missing 2 required positional arguments: 'x_test' and 'y_test'

In [ ]:
print(sys.getsizeof(test_obu.learner.state_dict()))
# should be 1424? 

In [ ]:
print("=== Test Results After Training (full test set): ===")
test_obu.test(x_test, y_test)

In [ ]:
print("=== Test Results After Training (full test set): ===")
test_obu.test(x_sm_test, y_test)